# **Trabajo pr?ctico 1, IA explicable**
## Parte 2: Modelo lineal - Regresi?n log?stica
Juan Eizaguerri Serrano

Daniel Hern?ndez Mart?nez


IMPORTANTE: Este notebook requiere tener en la carpeta montada de Google Colab un directorio /data que contenga los ficheros train.csv y validation.csv proporcionados como material de la pr?ctica.

In [ ]:
#!pip install interpret six graphviz pydotplus

# Carga de librer?as

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

# Carga y limpieza de datos

In [ ]:
# Cargar conjuntos de datos
train_csv = pd.read_csv('data/train.csv')
validation_csv = pd.read_csv('data/validation.csv')
test_csv = pd.read_csv('data/test.csv')

# Mostrar tama?o de los conjuntos de datos
print('Train cargado, tama?o:', train_csv.shape)
print('Validation cargado, tama?o:', validation_csv.shape)
print('Test cargado, tama?o:', test_csv.shape)

# Separar datos y etiquetas
feature_cols = train_csv.columns.values.tolist()[1:]
X_train = pd.DataFrame(train_csv[feature_cols], columns=feature_cols)
y_train = train_csv['RiskPerformance']
X_validation = pd.DataFrame(validation_csv[feature_cols], columns=feature_cols)
y_validation = validation_csv['RiskPerformance']
X_test = pd.DataFrame(test_csv[feature_cols], columns=feature_cols)
y_test = test_csv['RiskPerformance']

# Calcular cantidad de valores 'good' y 'bad' en los conjuntos de datos
num_good_train = train_csv[train_csv['RiskPerformance'] == 'Good'].shape[0]
num_bad_train = train_csv[train_csv['RiskPerformance'] == 'Bad'].shape[0]
num_good_validation = validation_csv[validation_csv['RiskPerformance'] == 'Good'].shape[0]
num_bad_validation = validation_csv[validation_csv['RiskPerformance'] == 'Bad'].shape[0]
num_good_test = test_csv[test_csv['RiskPerformance'] == 'Good'].shape[0]
num_bad_test = test_csv[test_csv['RiskPerformance'] == 'Bad'].shape[0]

print('Proporci?n de buenos y malos en train:', num_good_train, num_bad_train)
print('Proporci?n de buenos y malos en validation:', num_good_validation, num_bad_validation)
print('Proporci?n de buenos y malos en test:', num_good_test, num_bad_test)



In [ ]:
X_train.corr()

In [ ]:
# Comprobar si hay valores nulos
print(train_csv.isnull().sum() + validation_csv.isnull().sum() + test_csv.isnull().sum())

In [ ]:
# Comprobar rangos de valores
print(X_train.describe())

# Normalizar los datos
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols)
X_validation = pd.DataFrame(scaler.transform(X_validation), columns=feature_cols)
X_test = pd.DataFrame(scaler.transform(X_test), columns=feature_cols)


# Entrenamiento del modelo

In [ ]:
# Modelo de regresi?n log?stica
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

model = LogisticRegression()
model.fit(X_train, y_train)
print(f'El algoritmo ha convergido en {model.n_iter_} iteraciones')

y_pred_train = model.predict(X_train)
print('Train accuracy:', accuracy_score(y_train, y_pred_train))
print('Confusion matrix:\n', confusion_matrix(y_train, y_pred_train))
print('='*50)
y_pred_validation = model.predict(X_validation)
print('Validation accuracy:', accuracy_score(y_validation, y_pred_validation))
print('Confusion matrix:\n', confusion_matrix(y_validation, y_pred_validation))

In [ ]:
# Probar distintos solvers y sacar accuracies
solvers = ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']
for solver in solvers:
    model = LogisticRegression(solver=solver)
    model.fit(X_train, y_train)
    y_pred_train = model.predict(X_train)
    print(f'Train accuracy ({solver}):', accuracy_score(y_train, y_pred_train))
    y_pred_validation = model.predict(X_validation)
    print(f'Validation accuracy ({solver}):', accuracy_score(y_validation, y_pred_validation))
    print('='*50)

In [ ]:
# Gr?fica de accuracy en funci?n de max_iter
import matplotlib.pyplot as plt

max_iters = range(1, 100, 5)
train_accuracies = []
validation_accuracies = []
for max_iter in max_iters:
    model = LogisticRegression(max_iter=max_iter)
    model.fit(X_train, y_train)
    y_pred_train = model.predict(X_train)
    train_accuracies.append(accuracy_score(y_train, y_pred_train))
    y_pred_validation = model.predict(X_validation)
    validation_accuracies.append(accuracy_score(y_validation, y_pred_validation))

plt.plot(max_iters, train_accuracies, label='Train')
plt.plot(max_iters, validation_accuracies, label='Validation')
plt.xlabel('max_iter')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Accuracy en funci?n de max_iter')
plt.show()

# Explicabilidad del modelo

## Explicabilidad global

In [ ]:
from interpret.glassbox import LogisticRegression
from interpret import show

# Crear y entrenar modelo
model = LogisticRegression()
model.fit(X_train, y_train)

# Mostrar explicaci?n
global_explanation = model.explain_global()
show(global_explanation)

## Interpretabilidad local

In [ ]:
# Mostrar la importancia de las variables en ejmplos
local_explanation = model.explain_local(X_validation, y_validation)
show(local_explanation)


# M?tricas del modelo final con datos de entrenamiento y test

In [ ]:
# Entrenar modelo de regresi?n log?stica
model = LogisticRegression()
model.fit(X_train, y_train)

# Predecir en el conjunto de entrenamiento y test
y_pred_train = model.predict(X_train)
y_pred_validation = model.predict(X_validation)
y_pred_test = model.predict(X_test)

# Calcular accuracy en el conjunto de entrenamiento y test
train_accuracy = round(accuracy_score(y_train, y_pred_train), 4)
test_accuracy = round(accuracy_score(y_test, y_pred_test), 4)

print('Train accuracy:', train_accuracy)
print('Test accuracy:', test_accuracy)

In [ ]:
from sklearn.metrics import roc_curve

roc_auc = roc_auc_score(y_test.map({'Bad': 0, 'Good': 1}), model.predict_proba(X_test)[:, 1])
fpr, tpr, _ = roc_curve(y_test.map({'Bad': 0, 'Good': 1}), model.predict_proba(X_test)[:, 1])

plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

# Calcular AUC
print('AUC:', round(roc_auc, 4))